In [9]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))

from cleaning import fill_missing_median, drop_missing, normalize_data

import pandas as pd
from pathlib import Path

# path of raw data
DATA_PATH = Path("..") / "data" / "raw" / "data.csv"

# dataset
df = pd.read_csv(DATA_PATH)

print(df.shape)        # rows, columns
display(df.head())     # first 5 rows
df.info()              # column types + null counts

# Fill missing with median
df_filled = fill_missing_median(df)

# Drop columns with >50% missing
df_dropped = drop_missing(df, threshold=0.5)

# Normalize numeric columns
df_normalized = normalize_data(df)

print("Original shape:", df.shape)
print("After filling:", df_filled.shape)
print("After dropping:", df_dropped.shape)
print("After normalization:", df_normalized.shape)



(4, 3)


,date,revenue,risk_score
0,2025-08-14,1200.0,0.35
1,2025-08-15,1525.0,0.30
2,2025-08-16,980.0,0.55
3,2025-08-17,1730.0,0.28


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        4 non-null      object 
 1   revenue     4 non-null      float64
 2   risk_score  4 non-null      float64
dtypes: float64(2), object(1)
memory usage: 228.0+ bytes
Original shape: (4, 3)
After filling: (4, 3)
After dropping: (4, 3)
After normalization: (4, 3)


In [10]:
# 'date' (string) dtype, convert to datetime
df_step = fill_missing_median(df)
df_step = drop_missing(df_step, threshold=0.5).copy()

if "date" in df_step.columns:
    try:
        df_step["date"] = pd.to_datetime(df_step["date"])
    except Exception as e:
        print("Date parse warning:", e)

# two versions: unscaled (readability) and scaled (for modeling)
df_clean_no_scale = df_step.copy()
df_clean_scaled   = normalize_data(df_step)

print("no-scale dtypes:\n", df_clean_no_scale.dtypes)
print("scaled dtypes:\n", df_clean_scaled.dtypes)


no-scale dtypes:
 date          datetime64[ns]
revenue              float64
risk_score           float64
dtype: object
scaled dtypes:
 date          datetime64[ns]
revenue              float64
risk_score           float64
dtype: object


In [11]:
from pathlib import Path
from datetime import datetime

OUT_DIR = Path("..") / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d-%H%M")

no_scale_csv = OUT_DIR / f"clean_no_scale_{ts}.csv"
scaled_parq  = OUT_DIR / f"clean_scaled_{ts}.parquet"

df_clean_no_scale.to_csv(no_scale_csv, index=False)
df_clean_scaled.to_parquet(scaled_parq, index=False)

no_scale_csv, scaled_parq


(WindowsPath('../data/processed/clean_no_scale_20250819-1133.csv'),
 WindowsPath('../data/processed/clean_scaled_20250819-1133.parquet'))

In [13]:
print("Original describe:")
display(df.describe(include="all"))

print("Clean (no scale) describe:")
display(df_clean_no_scale.describe(include="all"))

print("Scaled (numeric changes) preview:")
display(df_clean_scaled.head())


Original describe:


,date,revenue,risk_score
count,4,4.000000,4.000000
unique,4,NaN,NaN
top,2025-08-14,NaN,NaN
freq,1,NaN,NaN
mean,NaN,1358.750000,0.370000
std,NaN,333.725811,0.123558
min,NaN,980.000000,0.280000
25%,NaN,1145.000000,0.295000
50%,NaN,1362.500000,0.325000
75%,NaN,1576.250000,0.400000


Clean (no scale) describe:


,date,revenue,risk_score
count,4,4.000000,4.000000
mean,2025-08-15 12:00:00,1358.750000,0.370000
min,2025-08-14 00:00:00,980.000000,0.280000
25%,2025-08-14 18:00:00,1145.000000,0.295000
50%,2025-08-15 12:00:00,1362.500000,0.325000
75%,2025-08-16 06:00:00,1576.250000,0.400000
max,2025-08-17 00:00:00,1730.000000,0.550000
std,NaN,333.725811,0.123558


Scaled (numeric changes) preview:


,date,revenue,risk_score
0,2025-08-14,-0.549279,-0.186908
1,2025-08-15,0.575230,-0.654177
2,2025-08-16,-1.310485,1.682170
3,2025-08-17,1.284535,-0.841085
